# ASI background-subtraction diagnostic

Run all cells to compare background DN histograms and images for ARV, VEE, and BVR in green, red, and blue.
Set the UTC, elevation cutoff, and background buffer in the second code cell. Image overlays show the elevation cutoff and buffered background-selection boundary; contrast stretches are display-only.


In [ ]:
from pathlib import Path
import sys
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt

# Works with a kernel started in the repository or comparisons directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core/calibration.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from core import calibration
from core.tiff_utils import get_site_tiff_candidates, load_best_frame_from_tiffs
from core.constants import FRAME_INTERVAL_SECONDS_GREEN, FRAME_INTERVAL_SECONDS_RED


In [ ]:
DATE = "20260210"
UTC = "102400"
ELEVATION_CUTOFF_DEG = calibration.BACKGROUND_ELEVATION_CUTOFF_DEG  # Change cutoff here; rerun this cell and all below.
EDGE_BUFFER_PX = calibration.BACKGROUND_EDGE_BUFFER_PX       # Single buffer setting for every calculation and plot.

assert np.isfinite(EDGE_BUFFER_PX) and EDGE_BUFFER_PX >= 0
assert np.isfinite(ELEVATION_CUTOFF_DEG)
TARGET = dt.datetime.strptime(DATE + UTC, "%Y%m%d%H%M%S")

def selection_label():
    return f"At/below {ELEVATION_CUTOFF_DEG:g}° + {EDGE_BUFFER_PX:g} px buffer"


## All cameras: selected elevation cutoff and buffer

Compare ARV/VEE/BVR × green/red/blue at the selected UTC using `ELEVATION_CUTOFF_DEG` and `EDGE_BUFFER_PX` from the second code cell. Change those settings there, then rerun that cell and all cells below it to refresh the masks, histograms, and image overlays. Plot titles display the selected values. The production histogram estimator is used; Gaussian curves are area-normalized for display.

In [ ]:
from core import generate_skymap

sites = ("ARV", "VEE", "BVR")
colors = ("green", "red", "blue")
experiment_results = {}
experiment_frames = {}
fig, axes = plt.subplots(3, 3, figsize=(16, 12), layout="constrained")
for row, site in enumerate(sites):
    for col, color in enumerate(colors):
        ax = axes[row, col]
        ax.set_title(f"{site} — {color}")
        ax.set_xlabel("DN")
        ax.set_ylabel("Probability density")
        try:
            # Read elevation directly: no geographic reprojection is needed.
            payload = getattr(generate_skymap, f"load_{site}")(color=color)
            elevation = np.asarray(payload[3], dtype=float)
            below_cutoff = np.isfinite(elevation) & (elevation <= ELEVATION_CUTOFF_DEG)
            sample_mask = calibration.buffered_corner_mask(below_cutoff, EDGE_BUFFER_PX)
            candidates = get_site_tiff_candidates(site, DATE, color, mission="GNEISS")
            if not candidates:
                raise FileNotFoundError("No TIFFs available")
            cadence = {"green": FRAME_INTERVAL_SECONDS_GREEN,
                       "red": FRAME_INTERVAL_SECONDS_RED, "blue": 0.90178}[color]
            image, _ = load_best_frame_from_tiffs(site, candidates, TARGET, cadence, color=color)
            image = np.asarray(image)
            if image.shape != sample_mask.shape:
                raise ValueError(f"Image/starmap shape mismatch: {image.shape} / {sample_mask.shape}")
            experiment_frames[(site, color)] = (image, elevation, below_cutoff, sample_mask)
            estimate = calibration.estimate_corner_background(image, sample_mask)
            samples = calibration.finite_values(image[sample_mask])
            bins = np.arange(np.floor(samples.min()), np.ceil(samples.max()) + 1)
            if bins.size < 3:
                bins = np.linspace(samples.min(), samples.max() + 1, 3)
            ax.hist(samples, bins=bins, density=True, alpha=.55, color=f"tab:{color}")
            if not estimate["fit_failed"] and estimate["sigma"] > 0:
                x = np.linspace(bins[0], bins[-1], 1500)
                y = calibration.gauss(x, 1/(np.sqrt(2*np.pi)*estimate["sigma"]),
                                      estimate["center"], estimate["sigma"])
                ax.plot(x, y, color="tab:orange", label="Gaussian shape")
            method = "median fallback" if estimate["fit_failed"] else "Gaussian center"
            ax.axvline(estimate["center"], color="black", label=f"{method}: {estimate['center']:.2f} DN")
            ax.axvline(estimate["median"], color="gray", linestyle="--", label=f"Median: {estimate['median']:.2f}")
            ax.set_title(f"{site} — {color} | N={samples.size:,} | σ={estimate['sigma']:.2f} DN")
            ax.legend(fontsize=8)
            experiment_results[(site, color)] = estimate
        except (FileNotFoundError, OSError, ValueError, RuntimeError) as exc:
            experiment_results[(site, color)] = {"error": str(exc)}
            ax.text(.5, .5, str(exc), transform=ax.transAxes, ha="center", va="center", wrap=True, fontsize=9)
            print(f"{site} {color}: {exc}")
fig.suptitle(f"{selection_label()} | requested UTC {TARGET}", fontsize=15)
plt.show()


## DN images: elevation cutoff and background boundary

Yellow marks the configured elevation contour, or the edge of the excluded region if a continuous contour is unavailable (a proxy). Magenta dashed marks the background-selection boundary after `EDGE_BUFFER_PX`; samples lie on its outer/corner side. The legend and title display the settings from the second code cell. These are the same frames and masks used in the histograms. Contrast stretches are display-only. ARV blue includes the detector column correction.

In [ ]:
from matplotlib.lines import Line2D

fig, axes = plt.subplots(3, 3, figsize=(17, 15), layout="constrained")
for row, site in enumerate(sites):
    for col, color in enumerate(colors):
        ax = axes[row, col]
        ax.set_title(f"{site} — {color}")
        ax.set_xlabel("Pixel column")
        ax.set_ylabel("Pixel row")
        if (site, color) not in experiment_frames:
            message = experiment_results.get((site, color), {}).get("error", "No frame loaded; run the histogram cell first")
            ax.text(.5, .5, message, transform=ax.transAxes, ha="center", va="center", wrap=True)
            continue
        image, elevation, below_cutoff, sample_mask = experiment_frames[(site, color)]
        finite = image[np.isfinite(image)]
        low, high = np.percentile(finite, [1, 99.7])
        im = ax.imshow(image, origin="upper", cmap="gray", vmin=low, vmax=high)
        fig.colorbar(im, ax=ax, label="Raw DN", shrink=.75)
        if np.any(np.isfinite(elevation) & (elevation < ELEVATION_CUTOFF_DEG)):
            ax.contour(np.ma.masked_invalid(elevation), levels=[ELEVATION_CUTOFF_DEG], colors="yellow", linewidths=1.5)
            horizon_label = f"{ELEVATION_CUTOFF_DEG:g}° elevation"
        else:
            ax.contour(below_cutoff.astype(float), levels=[.5], colors="yellow", linewidths=1.5)
            horizon_label = f"{ELEVATION_CUTOFF_DEG:g}° region boundary (proxy)"
        if np.any(sample_mask) and not np.all(sample_mask):
            ax.contour(sample_mask.astype(float), levels=[.5], colors="magenta", linewidths=1.5, linestyles="--")
        ax.legend(handles=[Line2D([], [], color="yellow", label=horizon_label),
                           Line2D([], [], color="magenta", linestyle="--", label=f"+{EDGE_BUFFER_PX:g} px background boundary")],
                  loc="upper right", fontsize=7, facecolor="#333333", labelcolor="white", framealpha=.85)
fig.suptitle(f"Raw DN | {selection_label()} | requested UTC {TARGET}", fontsize=15)
plt.show()
